## DuckDB with AWS S3 Example

This example shows how to access AWS S3 from DuckDB.

You can access MinIO, Cloudflare R2, and GCS from DuckDB in the same way as in this example.

### Setting up

In [ ]:
import os
from pathlib import Path

import duckdb

### Connecting to DuckDB database

In [ ]:
db = duckdb.connect(
    database=Path(os.environ["DUCKDB_DIR"]) / "example.db",
    read_only=False,
)

### Installing DuckDB extensions

In [ ]:
db.install_extension("httpfs")

db.load_extension("httpfs")

### Creating DuckDB secret

This example creates a secret to access AWS S3 from DuckDB using an AWS key pair.

If you want to access other storage services or use other authentication methods, see the following example and the [documentation](https://duckdb.org/docs/extensions/httpfs/s3api.html).


#### Examples


##### AWS S3 with Access Key Pair

```sql
CREATE OR REPLACE SECRET example (
    TYPE S3,
    REGION '<Your Region>',
    KEY_ID '<Your Access Key>',
    SECRET  '<Your Secret Access Key>'
);
```


##### AWS S3 with Credential Provider Chain

````sql
CREATE OR REPLACE SECRET example (
    TYPE S3,
    REGION 'us-east-1',
    PROVIDER CREDENTIAL_CHAIN,
    CHAIN 'env;config',
    PROFILE '<Your AWS Profile>'
);
````


##### GCS

> [!NOTE]
> DuckDB uses the S3 compatible APIs to access GCS.

```sql
CREATE OR REPLACE SECRET example (
    TYPE GCS,
    KEY_ID '<Your Access Key>',
    SECRET  '<Your Secret>'
);
```

In [ ]:
s = """
    CREATE OR REPLACE SECRET example (
        TYPE S3,
        REGION 'us-east-1',
        KEY_ID '%(s3_access_key_id)s',
        SECRET  '%(s3_secret_access_key)s',
        ENDPOINT 'localhost:%(s3_port)s',
        URL_STYLE 'path',
        USE_SSL FALSE
    );
""" % {
    "s3_access_key_id": os.environ["MINIO_ROOT_USER"],
    "s3_secret_access_key": os.environ["MINIO_ROOT_PASSWORD"],
    "s3_port": os.environ["MINIO_PORT"],
}
_ = db.execute(s)

### Reading files from AWS S3

In [ ]:
db.sql("SELECT * FROM READ_JSON('s3://default/examples/first.jsonl');")

### Creating table from files on AWS S3

In [ ]:
db.sql("""
    CREATE OR REPLACE TABLE tmp_duckdb_with_aws_s3 AS
        SELECT * FROM READ_JSON('s3://default/examples/first.jsonl');
""")

In [ ]:
db.sql("SELECT * FROM tmp_duckdb_with_aws_s3;")

### Cleaning up

In [ ]:
db.close()